# 03 – Feature Engineering for Suzuki Coupling Yield Prediction

This notebook walks through every step of the feature-engineering pipeline used to
convert a tabular reaction dataset into a machine-learning-ready numeric matrix.

---

## Context

We are solving a **reaction-yield regression** problem for **Suzuki–Miyaura cross-coupling** reactions.

| Role | Example | Description |
|---|---|---|
| **Reactant 1** | 6-Chloroquinoline | Electrophilic (aryl halide / pseudo-halide / boronate) partner |
| **Reactant 2** | 2a Boronic Acid | Nucleophilic (boronate) partner |
| **Ligand** | SPhos | Phosphine ligand that tunes the Pd catalyst |
| **Base** | K₃PO₄ | Activates the boronate for transmetalation |
| **Solvent** | THF | Reaction medium |
| **Target (y)** | `yield_uv` | UV-measured yield (0 – 100 %) |

The feature matrix **X** is built from two complementary representations:

1. **DRFP** (Differential Reaction FingerPrint) — encodes *bond changes* between reactants and products.
2. **RDKit molecular descriptors** — encode the physicochemical properties of ligand, base, and solvent.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import math

from drfp import DrfpEncoder                        # reaction fingerprints
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch  # used only to persist the scaler via torch.save


## 2. Reaction Component Dictionaries

Each reaction component is stored as a **SMILES string** (Simplified Molecular Input
Line Entry System), the standard text-based molecular representation.

We maintain one dictionary per component type.  This lets us map human-readable
column values (e.g. `"SPhos"`) to their SMILES programmatically without any joins.

> **Why SMILES?** SMILES are the universal input format for both the DRFP encoder
> and RDKit descriptor calculators, so a single lookup table covers all downstream needs.


In [ ]:
# ── Reactant 1: electrophilic coupling partners ─────────────────────────────
# Includes aryl chlorides, bromides, iodides, triflates, and boronate esters/trifluoroborates.
# The leaving group determines reactivity, so structural diversity here is crucial.
REACTANT_1_SMILES = {
    '6-chloroquinoline':                     'Clc1ccc2ncccc2c1',
    '6-Bromoquinoline':                      'Brc1ccc2ncccc2c1',
    '6-triflatequinoline':                   'FC(F)(F)S(=O)(=O)Oc1ccc2ncccc2c1',
    '6-Iodoquinoline':                       'Ic1ccc2ncccc2c1',
    '6-quinoline-boronic acid hydrochloride': '[Cl-].OB(O)c1ccc2[nH+]cccc2c1',
    'Potassium quinoline-6-trifluoroborate':  '[K+].F[B-](F)(F)c1ccc2ncccc2c1',
    '6-Quinolineboronic acid pinacol ester':  'CC1(C)OB(c2cc3cccnc3cc2)OC1(C)C',
}

# ── Reactant 2: nucleophilic coupling partners ───────────────────────────────
# Four boron-activation forms of the same core scaffold (pyrazole-methylbenzene).
# Different activation states affect reactivity/selectivity.
REACTANT_2_SMILES = {
    '2a, Boronic Acid':    'Cc1ccc2c(cnn2C2CCCCO2)c1B(O)O',
    '2b, Boronic Ester':   'Cc1ccc2c(cnn2C2CCCCO2)c1B1OC(C)(C)C(C)(C)O1',
    '2c, Trifluoroborate': 'Cc1ccc2c(cnn2C2CCCCO2)c1[B-](F)(F)F.[K+]',
    '2d, Bromide':         'Cc1ccc2c(cnn2C2CCCCO2)c1Br',
}

# ── Ligands: phosphine ligands for Pd catalyst ───────────────────────────────
# Ligand sterics/electronics dramatically influence oxidative addition and
# transmetalation rates. Includes mono- and bidentate phosphines.
LIGAND_SMILES = {
    'P(tBu)3':     'CC(C)(C)P(C(C)(C)C)C(C)(C)C',
    'P(Ph)3 ':     'P(c1ccccc1)(c1ccccc1)c1ccccc1',
    'AmPhos':      'CN(C)c1ccc(P(C(C)(C)C)C(C)(C)C)cc1',
    'P(Cy)3':      'P(C1CCCCC1)(C1CCCCC1)C1CCCCC1',
    'P(o-Tol)3':   'P(c1ccccc1C)(c1ccccc1C)c1ccccc1C',
    'CataCXium A': 'CCCCP(C12CC3CC(CC(C3)C1)C2)C12CC3CC(CC(C3)C1)C2',
    'SPhos':       'COc1cccc(OC)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1',
    'dtbpf':       'CC(C)(C)P(C1=CC=C[CH-]1)C(C)(C)C.CC(C)(C)P(C1=CC=C[CH-]1)C(C)(C)C.[Fe+2]',
    'XPhos':       'CC(C)c1cc(C(C)C)cc(C(C)C)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1',
    'dppf':        '[CH-]1C=CC(=C1)P(c2ccccc2)c3ccccc3.[CH-]1C=CC(=C1)P(c2ccccc2)c3ccccc3.[Fe+2]',
    'Xantphos':    'CC1(C)c2cccc(P(c3ccccc3)c3ccccc3)c2Oc2c(P(c3ccccc3)c3ccccc3)cccc21',
}

# ── Bases ────────────────────────────────────────────────────────────────────
# Inorganic and organic bases activate the boronate via transmetalation.
# Basicity and solubility vary greatly across this set.
BASE_SMILES = {
    'NaOH':   '[Na+].[OH-]',
    'NaHCO3': '[Na+].O=C([O-])O',
    'CsF':    '[Cs+].[F-]',
    'K3PO4':  '[K+].[K+].[K+].[O-]P(=O)([O-])[O-]',
    'KOH':    '[K+].[OH-]',
    'LiOtBu': '[Li+].CC(C)(C)[O-]',
    'Et3N':   'CCN(CC)CC',
}

# ── Solvents ─────────────────────────────────────────────────────────────────
# Polarity and protic/aprotic character affect catalyst stability and base solubility.
SOLVENT_SMILES = {
    'MeCN': 'CC#N', 'THF': 'C1CCOC1', 'DMF': 'CN(C)C=O',
    'MeOH': 'OC', 'MeOH/H2O_V2 9:1': 'CO.O', 'THF_V2': 'C1CCOC1',
}

print("Reactant 1 options:", len(REACTANT_1_SMILES))
print("Reactant 2 options:", len(REACTANT_2_SMILES))
print("Ligand options    :", len(LIGAND_SMILES))
print("Base options      :", len(BASE_SMILES))
print("Solvent options   :", len(SOLVENT_SMILES))


## 3. RDKit Molecular Descriptor Calculator

RDKit exposes **209 physicochemical descriptors** (MW, LogP, TPSA, H-bond donors/acceptors,
ring counts, etc.) computed directly from a SMILES string.

We set up the calculator **once** so we can reuse it efficiently.


In [ ]:
# Build the global descriptor list from RDKit
DESC_NAMES = [d[0] for d in Descriptors.descList]
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(DESC_NAMES)

print(f"Number of RDKit descriptors available: {len(DESC_NAMES)}")
print("First 10 descriptors:", DESC_NAMES[:10])


## 4. SMILES → Descriptor Vector

The helper function `smiles_to_descriptors` converts one SMILES string into a
numeric vector of RDKit descriptors.

### Multi-fragment SMILES (salts, counter-ions, metallocenes)

Several SMILES contain dots (`'.'`) representing **separate molecular fragments**
(e.g. `'[K+].F[B-](F)(F)F'`).  Many descriptors are undefined for ions or
metal complexes, so we extract only the **largest organic fragment** (by atom
count) before computing descriptors.

```
'[K+].F[B-](F)(F)F'  →  keep F[B-](F)(F)F  (4 heavy atoms vs 1 for K+)
```


In [ ]:
def smiles_to_descriptors(smi: str) -> np.ndarray:
    """
    Convert a (possibly multi-fragment) SMILES into a 1-D numpy array
    of RDKit molecular descriptors.

    Strategy for multi-fragment SMILES (salts, metallocenes):
      1. Split on '.' to get individual fragments.
      2. Select the fragment with the highest heavy-atom count —
         this is the chemically meaningful part.
      3. Compute all 209 RDKit descriptors on that fragment.

    Returns NaN-filled array if the molecule cannot be parsed.
    """
    parts = smi.split('.')
    # For each fragment, parse it; fall back to methane if parsing fails
    best = max(
        parts,
        key=lambda s: (Chem.MolFromSmiles(s) or Chem.MolFromSmiles('C')).GetNumAtoms()
    )
    mol = Chem.MolFromSmiles(best)
    if mol:
        return np.array(calculator.CalcDescriptors(mol), dtype=float)
    else:
        return np.full(len(DESC_NAMES), np.nan)

# Quick sanity check
sph_desc = smiles_to_descriptors(LIGAND_SMILES['SPhos'])
print(f"SPhos descriptor vector length : {len(sph_desc)}")
print(f"First 5 descriptor values      : {sph_desc[:5].round(3)}")
print(f"Any NaN?                       : {np.isnan(sph_desc).any()}")


## 5. Pre-computing Descriptor Caches

Rather than recomputing descriptors for every reaction row (which would be slow),
we build **lookup dictionaries** — one per component type — mapping component names
to their descriptor vectors.

`ZERO` is the fallback vector for rows where a component is absent (NaN in the CSV).


In [ ]:
# Pre-compute once; reuse for every row in the dataset
ligand_cache  = {k: smiles_to_descriptors(v) for k, v in LIGAND_SMILES.items()}
base_cache    = {k: smiles_to_descriptors(v) for k, v in BASE_SMILES.items()}
solvent_cache = {k: smiles_to_descriptors(v) for k, v in SOLVENT_SMILES.items()}

# Fallback for missing/NaN entries — a zero vector of the same length
ZERO = np.zeros(len(DESC_NAMES))

def get_desc(cache: dict, key) -> np.ndarray:
    """
    Safe lookup: returns ZERO vector if key is NaN (missing component),
    otherwise returns the cached descriptor array.
    """
    if isinstance(key, float) and math.isnan(key):
        return ZERO
    return cache[key]

print("Ligand  cache keys:", list(ligand_cache.keys()))
print("Base    cache keys:", list(base_cache.keys()))
print("Solvent cache keys:", list(solvent_cache.keys()))


## 6. Load the Cleaned Dataset

We load the pre-processed CSV (output of `01_data_cleaning.ipynb`).
Each row represents one reaction experiment with its measured UV yield.


In [ ]:
df = pd.read_csv("data/processed/suzuki_cleaned.csv")

print(f"Dataset shape : {df.shape}")
print(f"Columns       : {list(df.columns)}")
df.head()


## 7. Reaction Fingerprints (DRFP)

### What is DRFP?

The **Differential Reaction FingerPrint** (Probst et al., *Digital Discovery*, 2022)
encodes a **reaction** (not just a molecule) as a binary vector.

It works by:
1. Computing circular Morgan-like substructure hashes for **reactants** and **products** separately.
2. Taking the *symmetric difference* (XOR) of those sets.
3. Folding the result into a fixed-length bit vector.

This captures **what bonds are formed and broken** — exactly the information that
differentiates a successful reaction from a failed one.

### SMILES format required: `reactants >> products`

Because we only record reactants (no explicit product SMILES in the dataset),
we write the reaction as `"R1.R2>>"` — an empty product side still lets DRFP
compute the reactant-side fingerprint.

| Parameter | Value | Meaning |
|---|---|---|
| `n_folded_length` | 2048 | Output bit-vector length |
| `radius` | 3 | Max Morgan radius (neighbourhood depth) |
| `min_radius` | 0 | Include single-atom environments |
| `rings` | True | Encode ring membership as additional features |


In [ ]:
# Build reaction SMILES strings in the format  "R1.R2>>" (no product recorded)
reaction_smiles = (
    df["reactant_1"].map(REACTANT_1_SMILES) + "." +
    df["reactant_2"].map(REACTANT_2_SMILES) + ">>"
).tolist()

print("Example reaction SMILES:")
print(reaction_smiles[0])
print(f"\nTotal reactions: {len(reaction_smiles)}")


In [ ]:
# Encode all reactions — this is the most computationally intensive step
fps = DrfpEncoder.encode(
    reaction_smiles,
    n_folded_length=2048,   # fingerprint length (bits)
    radius=3,               # maximum neighbourhood radius
    min_radius=0,           # include radius-0 (single atoms)
    rings=True,             # include ring-membership information
    show_progress_bar=True,
)

drfp_array = np.array(fps)
print(f"\nDRFP array shape: {drfp_array.shape}")   # (n_reactions, 2048)
print(f"Data type       : {drfp_array.dtype}")
print(f"Bit density (mean bits set): {drfp_array.mean():.4f}")


## 8. Building Descriptor Arrays for Reaction Conditions

For each reaction row we look up the pre-computed descriptor vectors for the
**ligand**, **base**, and **solvent**.

This produces three 2-D arrays, each of shape `(n_reactions, 209)`.

> **Note on reactant descriptors:**  Reactant 1 and Reactant 2 are *already*
> captured by the DRFP fingerprint (their substructures appear in the reaction
> encoding), so we do **not** compute separate descriptor arrays for them.
> The conditions (ligand / base / solvent) are *not* part of the reaction SMILES
> and must be represented separately.


In [ ]:
ligand_array  = np.array([get_desc(ligand_cache,  l) for l in df["ligand"]])
base_array    = np.array([get_desc(base_cache,    b) for b in df["base"]])
solvent_array = np.array([get_desc(solvent_cache, s) for s in df["solvent"]])

print(f"Ligand  descriptor array : {ligand_array.shape}")
print(f"Base    descriptor array : {base_array.shape}")
print(f"Solvent descriptor array : {solvent_array.shape}")


## 9. Concatenating All Features & Cleaning

### Concatenation

We horizontally stack the four arrays to form the full feature matrix:

```
X_raw = [ DRFP (2048) | Ligand descriptors (209) | Base descriptors (209) | Solvent descriptors (209) ]
      → shape: (n_reactions, 2675)
```

### Feature Pruning

Three types of columns are **dropped** before modelling:

| Condition | Reason |
|---|---|
| Contains **NaN** | Molecule couldn't be parsed; can't feed NaN to most ML models |
| Contains **Inf** | Numeric overflow in descriptor calculation |
| **Zero variance** | Constant across all reactions → carries no information |


In [ ]:
# ── Concatenate all feature blocks ───────────────────────────────────────────
X_raw = np.concatenate(
    [drfp_array, ligand_array, base_array, solvent_array],
    axis=1
)
print(f"X_raw shape (before pruning): {X_raw.shape}")

# ── Identify and remove uninformative / invalid columns ──────────────────────
nan_mask      = np.isnan(X_raw).any(axis=0)    # columns with any NaN
inf_mask      = np.isinf(X_raw).any(axis=0)    # columns with any Inf
const_mask    = (np.var(X_raw, axis=0) == 0)   # zero-variance (constant) columns
drop          = nan_mask | inf_mask | const_mask

print(f"\nColumns with NaN      : {nan_mask.sum()}")
print(f"Columns with Inf      : {inf_mask.sum()}")
print(f"Constant columns      : {const_mask.sum()}")
print(f"Total dropped         : {drop.sum()}")

X = X_raw[:, ~drop]
print(f"\nX shape (after pruning): {X.shape}")


## 10. Target Variable

The raw yield column (`yield_uv`) is measured in percent (0–100).

We **normalise it to the [0, 1] range** by dividing by 100.  This is a common
practice in regression tasks: it keeps gradients well-scaled and makes loss
values more interpretable.


In [ ]:
y = df["yield_uv"].values / 100.0   # scale to [0, 1]

print(f"y shape      : {y.shape}")
print(f"y min / max  : {y.min():.3f} / {y.max():.3f}")
print(f"y mean ± std : {y.mean():.3f} ± {y.std():.3f}")


## 11. Train / Test Split

We hold out **20 %** of the data as a test set, stratified only by random seed
(no stratification by yield bucket, since yield is continuous).

`random_state=42` ensures reproducibility across runs.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

print(f"Training set : X_train {X_train.shape},  y_train {y_train.shape}")
print(f"Test set     : X_test  {X_test.shape},   y_test  {y_test.shape}")
print(f"Train / test split : {len(X_train) / len(X):.0%} / {len(X_test) / len(X):.0%}")


## 12. Feature Scaling (StandardScaler)

### Why scale?

The feature matrix combines:
- **Binary DRFP bits** (range 0–1)
- **RDKit descriptors** with wildly different scales (e.g. MW ≈ 400, TPSA ≈ 80, LogP ≈ 3)

Without scaling, gradient-based models (neural networks, SVR) and distance-based
models (KNN) are dominated by high-magnitude features.

### StandardScaler

$$z = \frac{x - \mu}{\sigma}$$

Each feature is centred (mean = 0) and scaled (std = 1).

### Data-leakage prevention

> **Critical rule:** fit the scaler **only on training data**, then apply the
> same transform to the test set.  Fitting on the combined dataset would leak
> test-set statistics into training — artificially inflating test performance.

```python
scaler.fit_transform(X_train)   # ← learns μ and σ from training rows ONLY
scaler.transform(X_test)        # ← applies learned μ and σ to test rows
```


In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)   # fit on train, transform train
X_test  = scaler.transform(X_test)        # transform test with train statistics

print(f"Post-scaling X_train mean  ≈ {X_train.mean():.6f}  (should be ~0)")
print(f"Post-scaling X_train std   ≈ {X_train.std():.6f}  (should be ~1)")
print(f"Post-scaling X_test  mean  ≈ {X_test.mean():.4f}  (may differ slightly from 0)")


## 13. Persisting Outputs

We save:

| File | Contents | Why |
|---|---|---|
| `X_train.npy` | Scaled training features | Input for model training |
| `X_test.npy` | Scaled test features | Input for model evaluation |
| `y_train.npy` | Training targets [0,1] | Input for model training |
| `y_test.npy` | Test targets [0,1] | Input for model evaluation |
| `X_features.npy` | Full (unscaled) feature matrix | Reproducibility / reuse |
| `y_target.npy` | Full target vector | Reproducibility / reuse |
| `scaler.pth` | Fitted `StandardScaler` | **Must** be reused at inference time |

> The scaler is saved with `torch.save` (uses Python's `pickle` under the hood)
> so it can be loaded back with `torch.load` in a PyTorch training notebook without
> additional dependencies.


In [ ]:
import os
os.makedirs("data/final",     exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

np.save("data/final/X_train.npy",    X_train)
np.save("data/final/X_test.npy",     X_test)
np.save("data/final/y_train.npy",    y_train)
np.save("data/final/y_test.npy",     y_test)

np.save("data/final/X_features.npy", X)          # full unscaled matrix
np.save("data/final/y_target.npy",   y)           # full target vector

torch.save(scaler, "data/processed/scaler.pth")  # save fitted scaler

print("All outputs saved.")
print(f"\nFinal feature matrix  — X : {X.shape}")
print(f"Training set          — X_train : {X_train.shape},  y_train : {y_train.shape}")
print(f"Test set              — X_test  : {X_test.shape},   y_test  : {y_test.shape}")


## 14. Summary

```
Raw data (CSV)
     │
     ├─ Reactant 1 + Reactant 2  ──►  Reaction SMILES  ──►  DRFP (2048-bit)
     │
     ├─ Ligand   ──►  RDKit descriptors (209)
     ├─ Base     ──►  RDKit descriptors (209)
     └─ Solvent  ──►  RDKit descriptors (209)
                                  │
                          Concatenate → X_raw (n, 2675)
                                  │
                       Drop NaN / Inf / constant cols
                                  │
                              X (n, d)    y = yield / 100
                                  │
                         80 / 20 train-test split
                                  │
                      StandardScaler (fit on train only)
                                  │
                    X_train  X_test  y_train  y_test  ✔
```

Next step → `04_model_training.ipynb`
